In [ ]:
# === auto-inserted: bev-solving src on path ===
import sys, pathlib
_root = pathlib.Path.cwd()
while _root != _root.parent and not (_root / 'src' / 'geometry.py').exists():
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))


# Train BEV v4 Cleaned Aug Smooth

Self-contained notebook for a cleaned `v4` branch with:
- merged `train + val` pool;
- empty-GT removal;
- smart dedup of near-static frames;
- test-matched grouped validation around 200 samples;
- compact rover embeddings with `Other=0`;
- safe RTMDet-lite camera augmentations;
- mild boundary-only smooth labeling;
- optional 2-GPU training via `nn.DataParallel`.

This notebook does not modify the source dataset folders. All caches and checkpoints are written only into `RUN_DIR`.


In [1]:
%load_ext autoreload
%autoreload 2

import os, copy, time, json, math, random
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image, ImageFile
from tqdm import tqdm

ImageFile.LOAD_TRUNCATED_IMAGES = True
os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'

DATA_TRAIN = Path('./autonomy_yandex_dataset_train/')
DATA_VAL   = Path('./autonomy_yandex_dataset_val/')
DATA_TEST  = Path('./autonomy_yandex_dataset_test/')

cfg = {
    'run_dir': './runs/v4_cleaned_aug_smooth',
    'img_hw': (384, 704),
    'batch_size': 8,
    'grad_accum': 4,
    'epochs': 20,
    'warmup_epochs': 3,
    'lr_backbone': 3e-5,
    'lr_head': 3e-4,
    'weight_decay': 1e-4,
    'embed_weight_decay': 1e-2,
    'pos_weight': 5.0,
    'num_workers': 4,
    'seed': 42,
    'ema_decay': 0.995,
    'val_target_size': 200,
    'min_rover_count': 30,
    'topk_rovers': 25,
    'rover_emb_dim': 8,
    'rover_cond_dim': 8,
    'mae_dedup_thr': 0.02,
    'dedup_camera': '/camera/inner/frontal/middle',
    'use_clean_cache': True,
    'post_scale_range': (0.90, 1.10),
    'rot_deg_range': (-3.0, 3.0),
    'flip_prob': 0.5,
    'drop_images_prob': 0.05,
    'max_dropped_images': 2,
    'smooth_sigma': 0.8,
    'smooth_kernel': 5,
    'smooth_boundary_width': 1,
    'use_dp_if_available': True,
}

RUN_DIR = Path(cfg['run_dir'])
RUN_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR = RUN_DIR / 'preproc_cache'
CACHE_DIR.mkdir(parents=True, exist_ok=True)

random.seed(cfg['seed'])
np.random.seed(cfg['seed'])
torch.manual_seed(cfg['seed'])

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
num_cuda = torch.cuda.device_count() if torch.cuda.is_available() else 0
print('device:', device)
print('cuda devices:', num_cuda)
if num_cuda:
    for i in range(num_cuda):
        print(i, torch.cuda.get_device_name(i))

with open(RUN_DIR / 'config.json', 'w') as f:
    json.dump({k: str(v) for k, v in cfg.items()}, f, indent=2)


device: cuda
cuda devices: 1
0 Tesla T4


In [ ]:
from src.geometry import load_info_with_root, resolve_info_path, resolve_row_path

CAMERA_NAMES = [
    '/camera/inner/frontal/middle',
    '/camera/inner/frontal/far',
    '/side/left/forward',
    '/side/right/forward',
]
INTRINSICS_NAMES = [n + '/intrinsic_params' for n in CAMERA_NAMES]
CAR2CAM_NAMES = [n + '/car_to_cam' for n in CAMERA_NAMES]
GT_NAME = 'gt_occupancy_grid'

CAM_IDX = {name: i for i, name in enumerate(CAMERA_NAMES)}
LEFT_CAM = '/side/left/forward'
RIGHT_CAM = '/side/right/forward'
LEFT_IDX = CAM_IDX[LEFT_CAM]
RIGHT_IDX = CAM_IDX[RIGHT_CAM]

BEV_H, BEV_W = 188, 126
BEV_RES = 0.8
X_RANGE = (0.0, BEV_H * BEV_RES)
Y_RANGE = (-BEV_W * BEV_RES / 2, BEV_W * BEV_RES / 2)
Z_LEVELS = (0.3, 1.0, 2.0, 3.0)

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)


In [ ]:
# Reusable code now lives in src/. See README.md.
from src.cleaning import build_img_hash, clean_merged_info, compute_gt_stats, smart_deduplicate


In [ ]:
# Reusable code now lives in src/. See README.md.
from src.splits import build_rover_vocab_from_train, encode_rover, make_test_matched_split_target


In [5]:
clean_info, dedup_report, clean_summary = clean_merged_info(
    DATA_TRAIN,
    DATA_VAL,
    cache_dir=CACHE_DIR,
    mae_thr=cfg['mae_dedup_thr'],
    dedup_camera=cfg['dedup_camera'],
    use_cache=cfg['use_clean_cache'],
)
print(json.dumps(clean_summary, indent=2, ensure_ascii=False))
display(clean_info.head(3))
if len(dedup_report):
    display(dedup_report.head(10))

train_idx, val_idx = make_test_matched_split_target(
    clean_info,
    DATA_TEST / 'info.csv',
    target_val_size=cfg['val_target_size'],
    seed=cfg['seed'],
    cache_path=CACHE_DIR / 'test_matched_val200_split.npz',
)
print('train_idx:', len(train_idx), 'val_idx:', len(val_idx))

train_info = clean_info.iloc[train_idx].reset_index(drop=True).copy()
val_info = clean_info.iloc[val_idx].reset_index(drop=True).copy()

rover_vocab, rover_stats = build_rover_vocab_from_train(
    train_info,
    min_count=cfg['min_rover_count'],
    topk=cfg['topk_rovers'],
)
rover_stats.to_csv(RUN_DIR / 'rover_embedding_stats.csv', index=False)
print('num rover classes including Other:', len(rover_vocab))
print('vocab head:', list(rover_vocab.items())[:10])
display(rover_stats.head(30))


{
  "merged_before_clean": 5000,
  "removed_empty_gt": 117,
  "after_empty_filter": 4883,
  "removed_by_dedup": 390,
  "clean_total": 4493,
  "dedup_groups": 173
}


,gt_occupancy_grid,header_ts,log_time,message_ts,/camera/inner/frontal/middle,/side/left/forward,/side/right/forward,/camera/inner/frontal/far,/camera/inner/frontal/middle/intrinsic_params,/camera/inner/frontal/middle/car_to_cam,/side/left/forward/intrinsic_params,/side/left/forward/car_to_cam,/side/right/forward/intrinsic_params,/side/right/forward/car_to_cam,/camera/inner/frontal/far/intrinsic_params,/camera/inner/frontal/far/car_to_cam,predicted_occupancy_grid,ride_date,ride_time,rover,scene_part_order,__data_root,__source_split,coverage,valid_count,pos_count
0,autonomy_yandex_dataset_train/static_grids/163...,1633857774533809000,12:18:58,1633857774533809000,autonomy_yandex_dataset_train/images/163385777...,autonomy_yandex_dataset_train/images/163385777...,autonomy_yandex_dataset_train/images/163385777...,autonomy_yandex_dataset_train/images/163385777...,autonomy_yandex_dataset_train/matrices/1633857...,autonomy_yandex_dataset_train/matrices/1633857...,autonomy_yandex_dataset_train/matrices/1633857...,autonomy_yandex_dataset_train/matrices/1633857...,autonomy_yandex_dataset_train/matrices/1633857...,autonomy_yandex_dataset_train/matrices/1633857...,autonomy_yandex_dataset_train/matrices/1633857...,autonomy_yandex_dataset_train/matrices/1633857...,autonomy_yandex_dataset_train/predicted_static...,2021-10-10,12:18:58,orvy,0,autonomy_yandex_dataset_train,train,0.061972,1468,602
1,autonomy_yandex_dataset_train/static_grids/163...,1636812143899937000,15:34:56,1636812143899937000,autonomy_yandex_dataset_train/images/163681214...,autonomy_yandex_dataset_train/images/163681214...,autonomy_yandex_dataset_train/images/163681214...,autonomy_yandex_dataset_train/images/163681214...,autonomy_yandex_dataset_train/matrices/1636812...,autonomy_yandex_dataset_train/matrices/1636812...,autonomy_yandex_dataset_train/matrices/1636812...,autonomy_yandex_dataset_train/matrices/1636812...,autonomy_yandex_dataset_train/matrices/1636812...,autonomy_yandex_dataset_train/matrices/1636812...,autonomy_yandex_dataset_train/matrices/1636812...,autonomy_yandex_dataset_train/matrices/1636812...,autonomy_yandex_dataset_train/predicted_static...,2021-11-13,15:34:56,shelly,0,autonomy_yandex_dataset_train,train,0.227077,5379,3752
2,autonomy_yandex_dataset_train/static_grids/163...,1633600207233930000,12:28:29,1633600207233930000,autonomy_yandex_dataset_train/images/163360020...,autonomy_yandex_dataset_train/images/163360020...,autonomy_yandex_dataset_train/images/163360020...,autonomy_yandex_dataset_train/images/163360020...,autonomy_yandex_dataset_train/matrices/1633600...,autonomy_yandex_dataset_train/matrices/1633600...,autonomy_yandex_dataset_train/matrices/1633600...,autonomy_yandex_dataset_train/matrices/1633600...,autonomy_yandex_dataset_train/matrices/1633600...,autonomy_yandex_dataset_train/matrices/1633600...,autonomy_yandex_dataset_train/matrices/1633600...,autonomy_yandex_dataset_train/matrices/1633600...,autonomy_yandex_dataset_train/predicted_static...,2021-10-07,12:28:29,orvy,0,autonomy_yandex_dataset_train,train,0.225008,5330,1264


,kept_row_id,group_size,members
0,2271,2,"[2271, 3683]"
1,4101,2,"[4473, 4101]"
2,1366,3,"[306, 2710, 1366]"
3,98,5,"[1721, 2885, 209, 3630, 98]"
4,4151,2,"[4824, 4151]"
5,1703,3,"[526, 277, 1703]"
6,4867,3,"[4332, 4815, 4867]"
7,2549,2,"[3137, 2549]"
8,2112,2,"[2112, 3216]"
9,2330,2,"[2938, 2330]"


train_idx: 4273 val_idx: 220
num rover classes including Other: 26
vocab head: [('__other__', 0), ('orvy', 1), ('shelly', 2), ('lerita', 3), ('ward', 4), ('ravine', 5), ('greben', 6), ('lucky', 7), ('miro', 8), ('benzon', 9)]


,rover,count,embedding_id,bucket
0,orvy,638,1,unique
1,shelly,496,2,unique
2,lerita,327,3,unique
3,ward,239,4,unique
4,ravine,208,5,unique
5,greben,194,6,unique
6,lucky,187,7,unique
7,miro,120,8,unique
8,benzon,114,9,unique
9,natelio,108,10,unique


In [6]:
def gaussian_kernel2d(kernel_size=5, sigma=0.8):
    ax = torch.arange(kernel_size, dtype=torch.float32) - kernel_size // 2
    xx, yy = torch.meshgrid(ax, ax, indexing='ij')
    kernel = torch.exp(-(xx ** 2 + yy ** 2) / (2 * sigma * sigma))
    kernel = kernel / kernel.sum()
    return kernel


def blur_mask_np(mask_np: np.ndarray, kernel_t: torch.Tensor) -> np.ndarray:
    x = torch.from_numpy(mask_np.astype(np.float32))[None, None]
    k = kernel_t[None, None]
    pad = kernel_t.shape[-1] // 2
    y = F.conv2d(x, k, padding=pad)
    return y.squeeze().numpy()


def dilate_binary_np(mask_np: np.ndarray, width: int = 1) -> np.ndarray:
    if width <= 0:
        return mask_np.astype(bool)
    x = torch.from_numpy(mask_np.astype(np.float32))[None, None]
    k = torch.ones((1, 1, 2 * width + 1, 2 * width + 1), dtype=torch.float32)
    y = F.conv2d(x, k, padding=width)
    return (y.squeeze().numpy() > 0)


In [7]:
class BEVDatasetV4CleanAugSmooth(Dataset):
    def __init__(self, info_df: pd.DataFrame, mode: str = 'train',
                 img_hw=(384, 704), aug: bool = False,
                 rover_vocab: dict | None = None,
                 post_scale_range=(0.90, 1.10),
                 rot_deg_range=(-3.0, 3.0),
                 flip_prob=0.5,
                 drop_images_prob=0.05,
                 max_dropped_images=2,
                 smooth_sigma=0.8,
                 smooth_kernel=5,
                 smooth_boundary_width=1):
        self.info = info_df.reset_index(drop=True).copy()
        self.mode = mode
        self.img_hw = img_hw
        self.aug = aug and (mode == 'train')
        self.rover_vocab = rover_vocab or {'__other__': 0}
        self.normalize = transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
        self.post_scale_range = post_scale_range
        self.rot_deg_range = rot_deg_range
        self.flip_prob = flip_prob
        self.drop_images_prob = drop_images_prob
        self.max_dropped_images = max_dropped_images
        self.smooth_sigma = smooth_sigma
        self.smooth_kernel = smooth_kernel
        self.smooth_boundary_width = smooth_boundary_width
        self.gauss_kernel = gaussian_kernel2d(smooth_kernel, smooth_sigma)

    def __len__(self):
        return len(self.info)

    def _resolve_path(self, row: pd.Series, key: str) -> Path:
        return resolve_info_path(Path(row['__data_root']), row[key])

    def _letterbox(self, img: Image.Image):
        src_W, src_H = img.size
        H_t, W_t = self.img_hw
        s = min(W_t / src_W, H_t / src_H)
        new_W, new_H = int(round(src_W * s)), int(round(src_H * s))
        img_resized = img.resize((new_W, new_H), Image.BILINEAR)
        canvas = Image.new('RGB', (W_t, H_t), (0, 0, 0))
        pad_x = (W_t - new_W) // 2
        pad_y = (H_t - new_H) // 2
        canvas.paste(img_resized, (pad_x, pad_y))
        return canvas, s, pad_x, pad_y

    def _affine_params(self):
        if not self.aug:
            return 1.0, 0.0
        scale = random.uniform(*self.post_scale_range)
        rot_deg = random.uniform(*self.rot_deg_range)
        return scale, rot_deg

    def _apply_affine_canvas(self, canvas: Image.Image, scale: float, rot_deg: float):
        H_t, W_t = self.img_hw
        out = canvas
        if abs(scale - 1.0) > 1e-6:
            sW, sH = int(round(W_t * scale)), int(round(H_t * scale))
            scaled = out.resize((sW, sH), Image.BILINEAR)
            if scale >= 1.0:
                dx = random.randint(0, sW - W_t)
                dy = random.randint(0, sH - H_t)
                out = scaled.crop((dx, dy, dx + W_t, dy + H_t))
            else:
                out = Image.new('RGB', (W_t, H_t), (0, 0, 0))
                dx = (W_t - sW) // 2
                dy = (H_t - sH) // 2
                out.paste(scaled, (dx, dy))
        else:
            dx = 0
            dy = 0

        if abs(rot_deg) > 1e-6:
            out = out.rotate(rot_deg, resample=Image.BILINEAR, expand=False, fillcolor=(0, 0, 0))
        return out, dx, dy

    def _make_affine_matrix(self, scale: float, rot_deg: float, dx: float, dy: float):
        H_t, W_t = self.img_hw
        cx = (W_t - 1) / 2.0
        cy = (H_t - 1) / 2.0

        T1 = np.array([[1, 0, -cx], [0, 1, -cy], [0, 0, 1]], dtype=np.float32)
        S = np.array([[scale, 0, 0], [0, scale, 0], [0, 0, 1]], dtype=np.float32)
        T2 = np.array([[1, 0, cx - dx], [0, 1, cy - dy], [0, 0, 1]], dtype=np.float32)
        theta = np.deg2rad(rot_deg)
        c = np.cos(theta).astype(np.float32)
        s = np.sin(theta).astype(np.float32)
        R = np.array([[c, -s, 0], [s, c, 0], [0, 0, 1]], dtype=np.float32)
        return T2 @ R @ S @ T1

    def _load_one_camera(self, row: pd.Series, cam_key: str, intr_key: str, c2c_key: str,
                         scale: float, rot_deg: float, flip_lr: bool):
        img = Image.open(self._resolve_path(row, cam_key)).convert('RGB')
        intr_path = self._resolve_path(row, intr_key)
        car2cam_path = self._resolve_path(row, c2c_key)

        canvas, s0, pad_x, pad_y = self._letterbox(img)
        canvas, dx, dy = self._apply_affine_canvas(canvas, scale, rot_deg)

        if flip_lr:
            canvas = canvas.transpose(Image.FLIP_LEFT_RIGHT)

        arr = np.array(canvas)
        if arr.ndim == 2:
            arr = np.stack([arr, arr, arr], axis=-1)
        img_t = torch.from_numpy(arr).permute(2, 0, 1).float() / 255.0
        img_t = self.normalize(img_t)

        intr_full = np.load(intr_path)
        K = intr_full[:, :3].copy().astype(np.float32)
        K[0, 0] *= s0; K[0, 2] *= s0
        K[1, 1] *= s0; K[1, 2] *= s0
        K[0, 2] += pad_x
        K[1, 2] += pad_y

        A = self._make_affine_matrix(scale, rot_deg, dx, dy)
        K = A @ K

        H_t, W_t = self.img_hw
        if flip_lr:
            Fm = np.array([[-1, 0, W_t - 1], [0, 1, 0], [0, 0, 1]], dtype=np.float32)
            K = Fm @ K

        car2cam = np.load(car2cam_path).astype(np.float32)
        return img_t, K, car2cam

    def _swap_lr_if_needed(self, items, flip_lr: bool):
        if not flip_lr:
            return items
        items = list(items)
        items[LEFT_IDX], items[RIGHT_IDX] = items[RIGHT_IDX], items[LEFT_IDX]
        return items

    def _make_soft_targets(self, gt_hard: np.ndarray):
        valid = (gt_hard != 255)
        occ = (gt_hard == 1) & valid
        free = (gt_hard == 0) & valid

        occ_dil = dilate_binary_np(occ, self.smooth_boundary_width)
        free_dil = dilate_binary_np(free, self.smooth_boundary_width)
        boundary = (occ_dil & free_dil) & valid

        occ_blur = blur_mask_np(occ.astype(np.float32), self.gauss_kernel)
        occ_blur = np.clip(occ_blur, 0.0, 1.0)

        gt_soft = np.zeros_like(gt_hard, dtype=np.float32)
        gt_soft[free] = 0.0
        gt_soft[occ] = 1.0
        gt_soft[boundary] = occ_blur[boundary]
        gt_soft[~valid] = 0.0
        valid_mask = valid.astype(np.float32)
        return gt_soft, valid_mask

    def _drop_random_images(self, images: list[torch.Tensor]):
        if not self.aug:
            return images
        if random.random() >= self.drop_images_prob:
            return images
        n_drop = random.randint(1, min(self.max_dropped_images, len(images)))
        drop_ids = random.sample(range(len(images)), k=n_drop)
        images = list(images)
        for di in drop_ids:
            images[di] = torch.zeros_like(images[di])
        return images

    def _load_sample(self, idx: int):
        row = self.info.iloc[idx]
        scale, rot_deg = self._affine_params()
        flip_lr = self.aug and (random.random() < self.flip_prob)

        imgs, Ks, M = [], [], []
        for cam, intr, c2c in zip(CAMERA_NAMES, INTRINSICS_NAMES, CAR2CAM_NAMES):
            img_t, K, c = self._load_one_camera(row, cam, intr, c2c, scale=scale, rot_deg=rot_deg, flip_lr=flip_lr)
            imgs.append(img_t)
            Ks.append(torch.from_numpy(K))
            M.append(torch.from_numpy(c))

        imgs = self._swap_lr_if_needed(imgs, flip_lr)
        Ks = self._swap_lr_if_needed(Ks, flip_lr)
        M = self._swap_lr_if_needed(M, flip_lr)
        imgs = self._drop_random_images(imgs)

        out = {
            'images': torch.stack(imgs, dim=0),
            'intrinsics': torch.stack(Ks, dim=0),
            'car2cams': torch.stack(M, dim=0),
            'rover_id': torch.tensor(encode_rover(row.get('rover', '__other__'), self.rover_vocab), dtype=torch.long),
            'info_idx': idx,
        }
        if self.mode == 'test':
            return out

        gt_hard = np.load(self._resolve_path(row, GT_NAME)).squeeze()
        gt_hard = np.where(gt_hard < 0, 255, gt_hard).astype(np.int64)
        gt_soft, valid_mask = self._make_soft_targets(gt_hard)

        out['gt_hard'] = torch.from_numpy(gt_hard).unsqueeze(0)
        out['gt_soft'] = torch.from_numpy(gt_soft).unsqueeze(0)
        out['valid_mask'] = torch.from_numpy(valid_mask).unsqueeze(0)
        return out

    def __getitem__(self, idx: int):
        max_tries = 5
        last_err = None
        for k in range(max_tries):
            try_idx = (idx + k) % len(self.info)
            try:
                return self._load_sample(try_idx)
            except (OSError, ValueError, FileNotFoundError) as e:
                last_err = e
                continue
        raise RuntimeError(f'Failed to load sample after {max_tries} tries from idx={idx}: {last_err}')


In [ ]:
from src.models.backbones import _ResNet50Backbone
from src.models.decoder import SmallUNet, _UNetBlock


class MultiCamBEVv4CleanAugSmooth(nn.Module):
    def __init__(self, num_rover_classes: int,
                 rover_emb_dim: int = 8,
                 rover_cond_dim: int = 8,
                 n_cameras: int = 4,
                 freeze_backbone: bool = False):
        super().__init__()
        self.n_cameras = n_cameras
        self.bev_h, self.bev_w = BEV_H, BEV_W
        self.x_range, self.y_range = X_RANGE, Y_RANGE
        self.z_levels = Z_LEVELS
        self.rover_cond_dim = rover_cond_dim

        self.backbone = _ResNet50Backbone(pretrained=True)
        if freeze_backbone:
            for p in self.backbone.parameters():
                p.requires_grad = False

        self.feat_proj = nn.Conv2d(128, 64, 1)
        self.register_buffer('ego_voxels', self._make_ego_voxels(), persistent=False)

        self.rover_embed = nn.Embedding(num_rover_classes, rover_emb_dim)
        nn.init.normal_(self.rover_embed.weight, std=0.02)
        self.rover_mlp = nn.Sequential(
            nn.Linear(rover_emb_dim, 16),
            nn.ReLU(inplace=True),
            nn.Linear(16, rover_cond_dim),
            nn.ReLU(inplace=True),
        )

        in_c = 64 * len(self.z_levels) + rover_cond_dim
        self.bev_decoder = SmallUNet(in_c=in_c, base_c=32, out_c=1)

    def _make_ego_voxels(self):
        xs = torch.linspace(self.x_range[0] + BEV_RES / 2, self.x_range[1] - BEV_RES / 2, self.bev_h)
        ys = torch.linspace(self.y_range[0] + BEV_RES / 2, self.y_range[1] - BEV_RES / 2, self.bev_w)
        zs = torch.tensor(self.z_levels, dtype=torch.float32)
        Z, X, Y = torch.meshgrid(zs, xs, ys, indexing='ij')
        ones = torch.ones_like(X)
        return torch.stack([X, Y, Z, ones], dim=-1)

    def forward(self, images, intrinsics, car2cams, rover_ids):
        B, N, C_, Hi, Wi = images.shape
        assert N == self.n_cameras

        feat = self.backbone(images.reshape(B * N, C_, Hi, Wi))
        feat = self.feat_proj(feat)
        Hf, Wf = feat.shape[-2:]
        feat = feat.reshape(B, N, 64, Hf, Wf)

        Z, H, W, _ = self.ego_voxels.shape
        V = Z * H * W
        voxels = self.ego_voxels.reshape(-1, 4).unsqueeze(0).unsqueeze(0).expand(B, N, -1, -1)

        p_cam = torch.einsum('bnij,bnvj->bniv', car2cams, voxels)
        p_cam_3d = p_cam[:, :, :3]
        uv_homog = torch.einsum('bnij,bnjv->bniv', intrinsics, p_cam_3d)
        z = uv_homog[:, :, 2]
        u = uv_homog[:, :, 0] / z.clamp(min=1e-3)
        v = uv_homog[:, :, 1] / z.clamp(min=1e-3)

        u_n = 2.0 * u / Wi - 1.0
        v_n = 2.0 * v / Hi - 1.0
        valid = (z > 0.1) & (u_n.abs() <= 1.0) & (v_n.abs() <= 1.0)

        grid = torch.stack([u_n, v_n], dim=-1).reshape(B * N, V, 1, 2)
        sampled = F.grid_sample(
            feat.reshape(B * N, 64, Hf, Wf),
            grid,
            mode='bilinear',
            padding_mode='zeros',
            align_corners=False,
        )
        sampled = sampled.squeeze(-1).reshape(B, N, 64, V)

        valid_f = valid.float().unsqueeze(2)
        sampled = sampled * valid_f
        agg = sampled.sum(dim=1) / valid_f.sum(dim=1).clamp(min=1.0)
        agg = agg.reshape(B, 64, Z, H, W).reshape(B, 64 * Z, H, W)

        rover_feat = self.rover_mlp(self.rover_embed(rover_ids)).view(B, self.rover_cond_dim, 1, 1)
        rover_map = rover_feat.expand(-1, -1, H, W)
        agg = torch.cat([agg, rover_map], dim=1)
        return self.bev_decoder(agg)


In [ ]:
from src.eval import evaluate_iou
from src.losses import _lovasz_grad, lovasz_hinge_flat
from src.metrics import iou_binary_batch, streaming_threshold_sweep
from src.utils.training import unwrap_model, update_ema

class CompoundLossV2Soft(nn.Module):
    def __init__(self, pos_weight: float = 5.0,
                 weight_bce: float = 0.5,
                 weight_dice: float = 0.3,
                 weight_lovasz: float = 0.2):
        super().__init__()
        self.register_buffer('pos_weight', torch.tensor([pos_weight]))
        self.weight_bce = weight_bce
        self.weight_dice = weight_dice
        self.weight_lovasz = weight_lovasz

    def forward(self, logits: torch.Tensor, gt_soft: torch.Tensor, gt_hard: torch.Tensor, valid_mask: torch.Tensor):
        valid = valid_mask > 0.5

        bce_per = F.binary_cross_entropy_with_logits(logits, gt_soft, pos_weight=self.pos_weight, reduction='none')
        bce = (bce_per * valid.float()).sum() / valid.float().sum().clamp(min=1.0)

        prob = torch.sigmoid(logits) * valid.float()
        gt_d = gt_soft * valid.float()
        inter = (prob * gt_d).sum(dim=(1, 2, 3))
        denom = prob.sum(dim=(1, 2, 3)) + gt_d.sum(dim=(1, 2, 3))
        dice = (1.0 - (2 * inter + 1) / (denom + 1)).mean()

        gt_h = (gt_hard == 1).float()
        lov_logits = logits[valid]
        lov_gt = gt_h[valid]
        lov = lovasz_hinge_flat(lov_logits, lov_gt) if lov_logits.numel() > 0 else logits.sum() * 0.0

        total = self.weight_bce * bce + self.weight_dice * dice + self.weight_lovasz * lov
        parts = {'bce': bce.item(), 'dice': dice.item(), 'lovasz': lov.item()}
        return total, parts



In [10]:
ds_train_warm = BEVDatasetV4CleanAugSmooth(
    train_info, mode='train', img_hw=cfg['img_hw'], aug=False, rover_vocab=rover_vocab,
    post_scale_range=cfg['post_scale_range'], rot_deg_range=cfg['rot_deg_range'], flip_prob=cfg['flip_prob'],
    drop_images_prob=cfg['drop_images_prob'], max_dropped_images=cfg['max_dropped_images'],
    smooth_sigma=cfg['smooth_sigma'], smooth_kernel=cfg['smooth_kernel'], smooth_boundary_width=cfg['smooth_boundary_width'],
)
ds_train = BEVDatasetV4CleanAugSmooth(
    train_info, mode='train', img_hw=cfg['img_hw'], aug=True, rover_vocab=rover_vocab,
    post_scale_range=cfg['post_scale_range'], rot_deg_range=cfg['rot_deg_range'], flip_prob=cfg['flip_prob'],
    drop_images_prob=cfg['drop_images_prob'], max_dropped_images=cfg['max_dropped_images'],
    smooth_sigma=cfg['smooth_sigma'], smooth_kernel=cfg['smooth_kernel'], smooth_boundary_width=cfg['smooth_boundary_width'],
)
ds_val = BEVDatasetV4CleanAugSmooth(
    val_info, mode='val', img_hw=cfg['img_hw'], aug=False, rover_vocab=rover_vocab,
    post_scale_range=cfg['post_scale_range'], rot_deg_range=cfg['rot_deg_range'], flip_prob=cfg['flip_prob'],
    drop_images_prob=cfg['drop_images_prob'], max_dropped_images=cfg['max_dropped_images'],
    smooth_sigma=cfg['smooth_sigma'], smooth_kernel=cfg['smooth_kernel'], smooth_boundary_width=cfg['smooth_boundary_width'],
)

loader_warm = DataLoader(ds_train_warm, batch_size=cfg['batch_size'], shuffle=True,
                         num_workers=cfg['num_workers'], pin_memory=(device.type == 'cuda'), drop_last=True)
loader_train = DataLoader(ds_train, batch_size=cfg['batch_size'], shuffle=True,
                          num_workers=cfg['num_workers'], pin_memory=(device.type == 'cuda'), drop_last=True)
loader_val = DataLoader(ds_val, batch_size=cfg['batch_size'], shuffle=False,
                        num_workers=cfg['num_workers'], pin_memory=(device.type == 'cuda'))

sample = ds_train[0]
for k, v in sample.items():
    if isinstance(v, torch.Tensor):
        print(k, tuple(v.shape), v.dtype)
    else:
        print(k, type(v), v)


images (4, 3, 384, 704) torch.float32
intrinsics (4, 3, 3) torch.float32
car2cams (4, 4, 4) torch.float32
rover_id () torch.int64
info_idx <class 'int'> 0
gt_hard (1, 188, 126) torch.int64
gt_soft (1, 188, 126) torch.float32
valid_mask (1, 188, 126) torch.float32


In [11]:
base_model = MultiCamBEVv4CleanAugSmooth(
    num_rover_classes=len(rover_vocab),
    rover_emb_dim=cfg['rover_emb_dim'],
    rover_cond_dim=cfg['rover_cond_dim'],
).to(device)

if cfg['use_dp_if_available'] and device.type == 'cuda' and torch.cuda.device_count() > 1:
    print('Wrapping model with DataParallel on', torch.cuda.device_count(), 'GPUs')
    model = nn.DataParallel(base_model)
else:
    model = base_model

core_model = unwrap_model(model)
backbone_params, embed_params, other_params = [], [], []
for name, p in core_model.named_parameters():
    if not p.requires_grad:
        continue
    if name.startswith('backbone.'):
        backbone_params.append(p)
    elif name.startswith('rover_embed.'):
        embed_params.append(p)
    else:
        other_params.append(p)

optimizer = torch.optim.AdamW([
    {'params': backbone_params, 'lr': cfg['lr_backbone'], 'weight_decay': cfg['weight_decay']},
    {'params': other_params, 'lr': cfg['lr_head'], 'weight_decay': cfg['weight_decay']},
    {'params': embed_params, 'lr': cfg['lr_head'], 'weight_decay': cfg['embed_weight_decay']},
])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg['epochs'], eta_min=1e-6)
loss_fn = CompoundLossV2Soft(
    pos_weight=cfg['pos_weight'],
    weight_bce=0.3,
    weight_dice=0.3,
    weight_lovasz=0.40,
).to(device)
scaler = torch.cuda.amp.GradScaler(enabled=(device.type == 'cuda'))

ema_model = copy.deepcopy(core_model).to(device).eval()
for p in ema_model.parameters():
    p.requires_grad = False

print('trainable params M:', sum(p.numel() for p in core_model.parameters() if p.requires_grad) / 1e6)


Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /tmp/xdg_cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth
100%|██████████| 97.8M/97.8M [00:01<00:00, 65.5MB/s]


trainable params M: 3.484457


In [12]:
cfg['batch_size'] = 12

In [15]:
RUN_DIR = Path('runs/v4_cleaned_aug_smooth_weight_loss')

In [17]:
thresholds = [round(x, 2) for x in np.arange(0.30, 0.81, 0.02)]
log = []
best_iou = -1.0
best_ema_iou = -1.0
start = time.time()

for epoch in range(cfg['epochs']):
    model.train()
    loader = loader_warm if epoch < cfg['warmup_epochs'] else loader_train
    phase = 'WARM' if epoch < cfg['warmup_epochs'] else 'AUG'

    losses = []
    optimizer.zero_grad(set_to_none=True)
    pbar = tqdm(loader, desc=f'ep{epoch:02d} [{phase}]')
    for step, batch in enumerate(pbar):
        images = batch['images'].to(device, non_blocking=True)
        intr = batch['intrinsics'].to(device, non_blocking=True)
        c2c = batch['car2cams'].to(device, non_blocking=True)
        rover_id = batch['rover_id'].to(device, non_blocking=True)
        gt_hard = batch['gt_hard'].to(device, non_blocking=True)
        gt_soft = batch['gt_soft'].to(device, non_blocking=True)
        valid_mask = batch['valid_mask'].to(device, non_blocking=True)

        with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):
            logits = model(images, intr, c2c, rover_id)
            loss, parts = loss_fn(logits, gt_soft, gt_hard, valid_mask)
            loss = loss / cfg['grad_accum']

        scaler.scale(loss).backward()
        if (step + 1) % cfg['grad_accum'] == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            update_ema(ema_model, model, cfg['ema_decay'])

        losses.append(loss.item() * cfg['grad_accum'])
        if step % 20 == 0:
            pbar.set_postfix(loss=f'{np.mean(losses[-50:]):.3f}', bce=f"{parts['bce']:.2f}")

    scheduler.step()

    val_iou_05 = evaluate_iou(model, loader_val, threshold=0.5)
    val_iou_05_ema = evaluate_iou(ema_model, loader_val, threshold=0.5)
    elapsed = (time.time() - start) / 60

    row = {
        'epoch': epoch,
        'loss': float(np.mean(losses)),
        'val_iou_05': float(val_iou_05),
        'val_iou_05_ema': float(val_iou_05_ema),
        'minutes': float(elapsed),
    }

    if epoch % 2 == 0 or epoch == cfg['epochs'] - 1:
        sweep_model = streaming_threshold_sweep(model, loader_val, thresholds)
        best_t_model, best_iou_model = max(sweep_model.items(), key=lambda kv: kv[1])
        sweep_ema = streaming_threshold_sweep(ema_model, loader_val, thresholds)
        best_t_ema, best_iou_ema = max(sweep_ema.items(), key=lambda kv: kv[1])
        row.update({
            'best_t_model': float(best_t_model),
            'best_iou_model': float(best_iou_model),
            'best_t_ema': float(best_t_ema),
            'best_iou_ema': float(best_iou_ema),
        })
        print(f"ep{epoch:02d} | loss {np.mean(losses):.3f} | val@0.5 {val_iou_05:.4f} | ema@0.5 {val_iou_05_ema:.4f} | best_t {best_t_model:.2f}/{best_t_ema:.2f} | best_iou {best_iou_model:.4f}/{best_iou_ema:.4f} | {elapsed:.1f}m")

        if best_iou_model > best_iou:
            best_iou = best_iou_model
            torch.save({
                'model_type': 'v4_cleaned_aug_smooth',
                'model': unwrap_model(model).state_dict(),
                'epoch': epoch,
                'best_iou': best_iou_model,
                'best_t': best_t_model,
                'rover_vocab': rover_vocab,
                'cfg': cfg,
            }, RUN_DIR / 'best.pt')
            print('  new best model:', best_iou_model)

        if best_iou_ema > best_ema_iou:
            best_ema_iou = best_iou_ema
            torch.save({
                'model_type': 'v4_cleaned_aug_smooth',
                'model': unwrap_model(model).state_dict(),
                'ema': ema_model.state_dict(),
                'epoch': epoch,
                'best_ema_iou': best_iou_ema,
                'best_t_ema': best_t_ema,
                'rover_vocab': rover_vocab,
                'cfg': cfg,
            }, RUN_DIR / 'ema_best.pt')
            print('  new best ema:', best_iou_ema)
    else:
        print(f"ep{epoch:02d} | loss {np.mean(losses):.3f} | val@0.5 {val_iou_05:.4f} | ema@0.5 {val_iou_05_ema:.4f} | {elapsed:.1f}m")

    log.append(row)
    pd.DataFrame(log).to_csv(RUN_DIR / 'log.csv', index=False)
    torch.save({
        'model_type': 'v4_cleaned_aug_smooth',
        'model': unwrap_model(model).state_dict(),
        'ema': ema_model.state_dict(),
        'epoch': epoch,
        'rover_vocab': rover_vocab,
        'cfg': cfg,
    }, RUN_DIR / 'last.pt')


threshold sweep: 100%|██████████| 28/28 [00:18<00:00,  1.54it/s]

ep04 | loss 0.884 | val@0.5 0.5186 | ema@0.5 0.5148 | best_t 0.44/0.56 | best_iou 0.5186/0.5157 | 38.7m


  new best ema: 0.5157044617337602


ep05 [AUG]: 100%|██████████| 534/534 [07:43<00:00,  1.15it/s, bce=1.20, loss=0.860]


ep05 | loss 0.874 | val@0.5 0.5172 | ema@0.5 0.4794 | 47.7m


...

ep17 | loss 0.816 | val@0.5 0.5306 | ema@0.5 0.5309 | 152.5m


threshold sweep: 100%|██████████| 28/28 [00:18<00:00,  1.53it/s]


ep18 | loss 0.813 | val@0.5 0.5302 | ema@0.5 0.5316 | best_t 0.60/0.54 | best_iou 0.5322/0.5317 | 160.9m


threshold sweep: 100%|██████████| 28/28 [00:19<00:00,  1.41it/s]


ep19 | loss 0.815 | val@0.5 0.5311 | ema@0.5 0.5313 | best_t 0.56/0.56 | best_iou 0.5313/0.5319 | 169.9m


In [22]:
import json
import hashlib
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm


In [23]:
EVAL_DIR = RUN_DIR / 'global_threshold_eval'
EVAL_DIR.mkdir(parents=True, exist_ok=True)

test_info = load_info_with_root(DATA_TEST, 'test')

ds_test = BEVDatasetV4CleanAugSmooth(
    test_info,
    mode='test',
    img_hw=cfg['img_hw'],
    aug=False,
    rover_vocab=rover_vocab,
    post_scale_range=cfg['post_scale_range'],
    rot_deg_range=cfg['rot_deg_range'],
    flip_prob=cfg['flip_prob'],
    drop_images_prob=cfg['drop_images_prob'],
    max_dropped_images=cfg['max_dropped_images'],
    smooth_sigma=cfg['smooth_sigma'],
    smooth_kernel=cfg['smooth_kernel'],
    smooth_boundary_width=cfg['smooth_boundary_width'],
)

loader_test = DataLoader(
    ds_test,
    batch_size=cfg['batch_size'],
    shuffle=False,
    num_workers=cfg['num_workers'],
    pin_memory=(device.type == 'cuda'),
)

thresholds = [round(x, 2) for x in np.arange(0.20, 0.86, 0.02)]
print('num thresholds:', len(thresholds))


num thresholds: 33


In [24]:
def load_eval_model_from_ckpt(ckpt_path: Path, use_ema: bool = True):
    ckpt = torch.load(ckpt_path, map_location=device)

    model_eval = MultiCamBEVv4CleanAugSmooth(
        num_rover_classes=len(rover_vocab),
        rover_emb_dim=cfg['rover_emb_dim'],
        rover_cond_dim=cfg['rover_cond_dim'],
    ).to(device)

    if use_ema and 'ema' in ckpt:
        state = ckpt['ema']
        state_name = 'ema'
    else:
        state = ckpt['model']
        state_name = 'model'

    missing, unexpected = model_eval.load_state_dict(state, strict=False)
    print('loaded:', ckpt_path.name, '| state:', state_name,
          '| missing:', len(missing), '| unexpected:', len(unexpected))
    return model_eval, ckpt


In [25]:
@torch.no_grad()
def collect_probs_for_loader_global(model, loader, info_df, split_name: str, cache_prefix: str):
    cache_probs = EVAL_DIR / f'{cache_prefix}_{split_name}_probs.pt'
    cache_gt = EVAL_DIR / f'{cache_prefix}_{split_name}_gt.pt'

    if cache_probs.exists():
        probs = torch.load(cache_probs, map_location='cpu')
        gt = torch.load(cache_gt, map_location='cpu') if cache_gt.exists() else None
        print(f'loaded cache: {cache_prefix} / {split_name}')
        return probs, gt

    model.eval()
    probs_list = []
    gt_list = []

    for batch in tqdm(loader, desc=f'collect probs [{cache_prefix} | {split_name}]'):
        images = batch['images'].to(device, non_blocking=True)
        intr = batch['intrinsics'].to(device, non_blocking=True)
        c2c = batch['car2cams'].to(device, non_blocking=True)
        rover_id = batch['rover_id'].to(device, non_blocking=True)

        with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):
            logits = model(images, intr, c2c, rover_id).float()

        probs = torch.sigmoid(logits).cpu().to(torch.float16)
        probs_list.append(probs)

        if 'gt_hard' in batch:
            gt_list.append(batch['gt_hard'].cpu().to(torch.uint8))

    probs = torch.cat(probs_list, dim=0)
    gt = torch.cat(gt_list, dim=0) if len(gt_list) else None

    torch.save(probs, cache_probs)
    if gt is not None:
        torch.save(gt, cache_gt)

    return probs, gt


In [26]:
def iou_for_probs(probs: torch.Tensor, gt_hard: torch.Tensor, threshold: float):
    valid = (gt_hard != 255)
    gt_b = ((gt_hard == 1) & valid)
    pred = ((probs > threshold) & valid)

    inter = (pred & gt_b).sum().item()
    union = (pred | gt_b).sum().item()
    return inter / max(union, 1)


In [27]:
CKPT_PATH = RUN_DIR / 'ema_best.pt' if (RUN_DIR / 'ema_best.pt').exists() else RUN_DIR / 'best.pt'
USE_EMA = True

model_eval, ckpt = load_eval_model_from_ckpt(CKPT_PATH, use_ema=USE_EMA)

val_probs, val_gt = collect_probs_for_loader_global(
    model_eval,
    loader_val,
    val_info,
    split_name='val',
    cache_prefix='global_eval',
)

test_probs, _ = collect_probs_for_loader_global(
    model_eval,
    loader_test,
    test_info,
    split_name='test',
    cache_prefix='global_eval',
)


loaded: ema_best.pt | state: ema | missing: 0 | unexpected: 0


collect probs [global_eval | val]: 100%|██████████| 28/28 [00:18<00:00,  1.51it/s]
collect probs [global_eval | test]: 100%|██████████| 167/167 [02:42<00:00,  1.03it/s]


In [28]:
rows = []
best_t = None
best_iou = -1.0

for t in thresholds:
    iou = iou_for_probs(val_probs, val_gt, t)
    rows.append({'threshold': float(t), 'iou': float(iou)})
    if iou > best_iou:
        best_iou = iou
        best_t = float(t)

thr_df = pd.DataFrame(rows)
display(thr_df)

print('best threshold:', best_t)
print('best iou:', best_iou)


,threshold,iou
0,0.20,0.486983
1,0.22,0.491673
2,0.24,0.496105
3,0.26,0.500385
4,0.28,0.504749
5,0.30,0.508660
6,0.32,0.512499
7,0.34,0.515764
8,0.36,0.518814
9,0.38,0.521611


best threshold: 0.64
best iou: 0.537059473401044


In [30]:
submit_thresholds = sorted(set([
    round(best_t - 0.06, 2),
    round(best_t - 0.04, 2),
    round(best_t - 0.02, 2),
    round(best_t, 2),
    round(best_t + 0.02, 2),
    round(best_t + 0.04, 2),
    round(best_t + 0.06, 2)
]))
submit_thresholds = [t for t in submit_thresholds if 0.05 <= t <= 0.95]

print('submit thresholds:', submit_thresholds)


submit thresholds: [0.58, 0.6, 0.62, 0.64, 0.66, 0.68, 0.7]


In [33]:
def make_submission_from_global_probs(test_probs, threshold, tag='global'):
    work_dir = Path('.')
    thr_tag = f'{threshold:.2f}'.replace('.', 'p')

    pred_dir = work_dir / f'predicted_static_grids_{tag}_{thr_tag}'
    pred_dir.mkdir(parents=True, exist_ok=True)

    for p in pred_dir.glob('*.npy'):
        p.unlink()

    preds = (test_probs > threshold).numpy().astype(np.int32)

    for i in tqdm(range(len(test_info)), desc=f'save npy t={threshold:.2f}'):
        rel_out_path = test_info.iloc[i]['predicted_occupancy_grid']
        out_name = Path(rel_out_path).name
        np.save(pred_dir / out_name, preds[i].reshape(1, BEV_H, BEV_W))

    sub_path = work_dir / f'submission_{RUN_DIR.name}_{tag}_t_{thr_tag}.zip'
    if sub_path.exists():
        sub_path.unlink()

    with zipfile.ZipFile(sub_path, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
        zf.write(DATA_TEST / 'info.csv', arcname='info.csv')
        for npy in sorted(pred_dir.glob('*.npy')):
            zf.write(npy, arcname=f'predicted_static_grids/{npy.name}')

    with zipfile.ZipFile(sub_path, 'r') as zf:
        bad = zf.testzip()
        assert bad is None, bad

    h = hashlib.sha256()
    with open(sub_path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)

    result = {
        'threshold': float(threshold),
        'submission': str(sub_path.resolve()),
        'size_mb': round(sub_path.stat().st_size / 1e6, 3),
        'sha256': h.hexdigest(),
    }
    print(json.dumps(result, indent=2, ensure_ascii=False))
    return result


In [36]:
submission_results = []

tag = 'ema_global' if USE_EMA and 'ema' in ckpt else 'best_global'

for t in [0.85]:
    print('\n' + '=' * 100)
    print(f'building submission for threshold={t:.2f}')
    result = make_submission_from_global_probs(test_probs, t, tag=tag)
    submission_results.append(result)



building submission for threshold=0.85


save npy t=0.85: 100%|██████████| 2000/2000 [00:43<00:00, 46.12it/s]


{
  "threshold": 0.85,
  "submission": "/home/jupyter/project/submission_v4_cleaned_aug_smooth_ema_global_t_0p85.zip",
  "size_mb": 2.365,
  "sha256": "ed418928202ba1a8cf7497e77d95311e9b7e5450da6207ed4f61859d729ec8b4"
}


In [37]:
print(1)

1


In [ ]:
ckpt = torch.load(RUN_DIR / 'ema_best.pt', map_location=device) if (RUN_DIR / 'ema_best.pt').exists() else torch.load(RUN_DIR / 'best.pt', map_location=device)
model_eval = MultiCamBEVv4CleanAugSmooth(
    num_rover_classes=len(rover_vocab),
    rover_emb_dim=cfg['rover_emb_dim'],
    rover_cond_dim=cfg['rover_cond_dim'],
).to(device)
state = ckpt['ema'] if 'ema' in ckpt else ckpt['model']
model_eval.load_state_dict(state, strict=False)

thresholds = [round(x, 2) for x in np.arange(0.20, 0.81, 0.02)]
iou_by_t = streaming_threshold_sweep(model_eval, loader_val, thresholds)
best_t, best_iou = max(iou_by_t.items(), key=lambda kv: kv[1])
print('best threshold:', best_t, 'best_iou:', best_iou)
for t, iou in iou_by_t.items():
    marker = ' <- best' if t == best_t else ''
    print(f't={t:.2f}  iou={iou:.4f}{marker}')


## Notes

- This notebook keeps the cleaned data pipeline and small rover embeddings from the cleaned `v4` branch.
- Added train-time RTMDet-lite augmentations in a geometry-safe form for camera-only BEV.
- Added mild boundary-only soft targets for training, while evaluation stays on hard GT.
- All caches and checkpoints are written only into `RUN_DIR`.


In [38]:
from torch.utils.data import Subset


In [39]:
SPECIALIST_ROVER = 'darron'

specialist_cfg = {
    'run_dir': str(RUN_DIR / f'specialist_{SPECIALIST_ROVER}'),
    'epochs': 6,
    'warmup_epochs': 1,
    'batch_size': 4,
    'val_batch_size': 2,
    'grad_accum': 2,
    'lr_backbone': 5e-6,
    'lr_head': 5e-5,
    'weight_decay': 1e-4,
    'embed_weight_decay': 1e-2,
    'ema_decay': 0.995,
    'num_workers': 2,
    'seed': 42,
    'holdout_frac': 0.2,
}

SPECIALIST_RUN_DIR = Path(specialist_cfg['run_dir'])
SPECIALIST_RUN_DIR.mkdir(parents=True, exist_ok=True)

print('specialist rover:', SPECIALIST_ROVER)
print('specialist run dir:', SPECIALIST_RUN_DIR)


specialist rover: darron
specialist run dir: runs/v4_cleaned_aug_smooth/specialist_darron


In [40]:
def make_rover_group_split(info_df: pd.DataFrame, holdout_frac=0.2, seed=42, group_col='ride_date'):
    rng = np.random.RandomState(seed)
    info_df = info_df.reset_index(drop=True).copy()

    if len(info_df) == 0:
        raise ValueError('empty rover dataframe')

    if group_col not in info_df.columns:
        idx = list(range(len(info_df)))
        rng.shuffle(idx)
        n_val = max(1, int(round(len(idx) * holdout_frac)))
        val_idx = sorted(idx[:n_val])
        train_idx = sorted(idx[n_val:])
        return train_idx, val_idx

    groups = list(info_df.groupby(group_col).groups.keys())
    rng.shuffle(groups)
    n_val_groups = max(1, int(round(len(groups) * holdout_frac)))
    val_groups = set(groups[:n_val_groups])

    train_idx, val_idx = [], []
    for i, row in info_df.iterrows():
        if row[group_col] in val_groups:
            val_idx.append(i)
        else:
            train_idx.append(i)

    if len(train_idx) == 0:
        train_idx = val_idx[:-1]
        val_idx = val_idx[-1:]

    if len(val_idx) == 0:
        val_idx = train_idx[-1:]
        train_idx = train_idx[:-1]

    return train_idx, val_idx


In [41]:
rover_all_info = clean_info[clean_info['rover'] == SPECIALIST_ROVER].reset_index(drop=True).copy()

print('total samples for rover:', len(rover_all_info))
display(rover_all_info[['rover', 'ride_date', 'message_ts', 'pos_count', 'valid_count']].head(20))

spec_train_idx, spec_val_idx = make_rover_group_split(
    rover_all_info,
    holdout_frac=specialist_cfg['holdout_frac'],
    seed=specialist_cfg['seed'],
    group_col='ride_date',
)

rover_train_info = rover_all_info.iloc[spec_train_idx].reset_index(drop=True).copy()
rover_val_info = rover_all_info.iloc[spec_val_idx].reset_index(drop=True).copy()

print('specialist train size:', len(rover_train_info))
print('specialist val size:', len(rover_val_info))
print('train ride_dates:', sorted(rover_train_info['ride_date'].astype(str).unique().tolist()))
print('val ride_dates:', sorted(rover_val_info['ride_date'].astype(str).unique().tolist()))


total samples for rover: 110


,rover,ride_date,message_ts,pos_count,valid_count
0,darron,2022-01-18,1642517758799896000,413,1582
1,darron,2023-08-19,1692426808100061000,1451,4888
2,darron,2022-07-25,1658763334000049000,4886,9207
3,darron,2022-07-24,1658678064200039000,3864,7355
4,darron,2022-03-30,1648655897199805000,415,1300
5,darron,2022-07-25,1658762280499986000,2133,5405
6,darron,2022-03-30,1648653293000140000,796,1736
7,darron,2023-08-07,1691385479999986000,3299,8047
8,darron,2022-03-30,1648651273399880000,3950,8189
9,darron,2025-02-06,1738825770500090000,1955,4376


specialist train size: 98
specialist val size: 12
train ride_dates: ['2021-10-28', '2021-10-29', '2021-11-04', '2021-11-06', '2021-11-08', '2021-11-09', '2021-11-10', '2021-11-15', '2021-11-16', '2022-01-18', '2022-01-20', '2022-01-24', '2022-01-25', '2022-01-26', '2022-01-28', '2022-03-30', '2022-07-24', '2022-07-25', '2023-07-05', '2023-08-02', '2023-08-07', '2023-08-08', '2023-08-12', '2023-08-13', '2023-08-14', '2023-08-19', '2023-08-22', '2023-08-23', '2023-08-26', '2023-08-30', '2024-12-14', '2025-01-10', '2025-01-14', '2025-02-06', '2025-02-11', '2025-03-03', '2025-05-20']
val ride_dates: ['2021-11-07', '2021-11-14', '2022-01-19', '2023-08-10', '2023-08-11', '2023-09-16', '2025-01-16', '2025-02-08', '2025-03-13']


In [42]:
ds_spec_train_warm = BEVDatasetV4CleanAugSmooth(
    rover_train_info,
    mode='train',
    img_hw=cfg['img_hw'],
    aug=False,
    rover_vocab=rover_vocab,
    post_scale_range=cfg['post_scale_range'],
    rot_deg_range=cfg['rot_deg_range'],
    flip_prob=cfg['flip_prob'],
    drop_images_prob=cfg['drop_images_prob'],
    max_dropped_images=cfg['max_dropped_images'],
    smooth_sigma=cfg['smooth_sigma'],
    smooth_kernel=cfg['smooth_kernel'],
    smooth_boundary_width=cfg['smooth_boundary_width'],
)

ds_spec_train = BEVDatasetV4CleanAugSmooth(
    rover_train_info,
    mode='train',
    img_hw=cfg['img_hw'],
    aug=True,
    rover_vocab=rover_vocab,
    post_scale_range=cfg['post_scale_range'],
    rot_deg_range=cfg['rot_deg_range'],
    flip_prob=cfg['flip_prob'],
    drop_images_prob=cfg['drop_images_prob'],
    max_dropped_images=cfg['max_dropped_images'],
    smooth_sigma=cfg['smooth_sigma'],
    smooth_kernel=cfg['smooth_kernel'],
    smooth_boundary_width=cfg['smooth_boundary_width'],
)

ds_spec_val = BEVDatasetV4CleanAugSmooth(
    rover_val_info,
    mode='val',
    img_hw=cfg['img_hw'],
    aug=False,
    rover_vocab=rover_vocab,
    post_scale_range=cfg['post_scale_range'],
    rot_deg_range=cfg['rot_deg_range'],
    flip_prob=cfg['flip_prob'],
    drop_images_prob=cfg['drop_images_prob'],
    max_dropped_images=cfg['max_dropped_images'],
    smooth_sigma=cfg['smooth_sigma'],
    smooth_kernel=cfg['smooth_kernel'],
    smooth_boundary_width=cfg['smooth_boundary_width'],
)

loader_spec_warm = DataLoader(
    ds_spec_train_warm,
    batch_size=specialist_cfg['batch_size'],
    shuffle=True,
    num_workers=specialist_cfg['num_workers'],
    pin_memory=(device.type == 'cuda'),
    drop_last=(len(ds_spec_train_warm) >= specialist_cfg['batch_size']),
)

loader_spec_train = DataLoader(
    ds_spec_train,
    batch_size=specialist_cfg['batch_size'],
    shuffle=True,
    num_workers=specialist_cfg['num_workers'],
    pin_memory=(device.type == 'cuda'),
    drop_last=(len(ds_spec_train) >= specialist_cfg['batch_size']),
)

loader_spec_val = DataLoader(
    ds_spec_val,
    batch_size=specialist_cfg['val_batch_size'],
    shuffle=False,
    num_workers=1,
    pin_memory=(device.type == 'cuda'),
)

sample = ds_spec_train[0]
for k, v in sample.items():
    if isinstance(v, torch.Tensor):
        print(k, tuple(v.shape), v.dtype)
    else:
        print(k, type(v), v)


images (4, 3, 384, 704) torch.float32
intrinsics (4, 3, 3) torch.float32
car2cams (4, 4, 4) torch.float32
rover_id () torch.int64
info_idx <class 'int'> 0
gt_hard (1, 188, 126) torch.int64
gt_soft (1, 188, 126) torch.float32
valid_mask (1, 188, 126) torch.float32


In [43]:
GLOBAL_CKPT_PATH = RUN_DIR / 'ema_best.pt' if (RUN_DIR / 'ema_best.pt').exists() else RUN_DIR / 'best.pt'
global_ckpt = torch.load(GLOBAL_CKPT_PATH, map_location=device)

specialist_base_model = MultiCamBEVv4CleanAugSmooth(
    num_rover_classes=len(rover_vocab),
    rover_emb_dim=cfg['rover_emb_dim'],
    rover_cond_dim=cfg['rover_cond_dim'],
).to(device)

global_state = global_ckpt['ema'] if 'ema' in global_ckpt else global_ckpt['model']
missing, unexpected = specialist_base_model.load_state_dict(global_state, strict=False)

print('loaded global ckpt for specialist')
print('checkpoint:', GLOBAL_CKPT_PATH)
print('missing:', len(missing), 'unexpected:', len(unexpected))


loaded global ckpt for specialist
checkpoint: runs/v4_cleaned_aug_smooth/ema_best.pt
missing: 0 unexpected: 0


In [45]:
if specialist_cfg.get('use_dp_if_available', False) and device.type == 'cuda' and torch.cuda.device_count() > 1:
    specialist_model = nn.DataParallel(specialist_base_model)
else:
    specialist_model = specialist_base_model

specialist_core = unwrap_model(specialist_model)

backbone_params, embed_params, other_params = [], [], []
for name, p in specialist_core.named_parameters():
    if not p.requires_grad:
        continue
    if name.startswith('backbone.'):
        backbone_params.append(p)
    elif name.startswith('rover_embed.'):
        embed_params.append(p)
    else:
        other_params.append(p)

specialist_optimizer = torch.optim.AdamW([
    {'params': backbone_params, 'lr': specialist_cfg['lr_backbone'], 'weight_decay': specialist_cfg['weight_decay']},
    {'params': other_params, 'lr': specialist_cfg['lr_head'], 'weight_decay': specialist_cfg['weight_decay']},
    {'params': embed_params, 'lr': specialist_cfg['lr_head'], 'weight_decay': specialist_cfg['embed_weight_decay']},
])

specialist_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    specialist_optimizer,
    T_max=specialist_cfg['epochs'],
    eta_min=1e-6,
)

specialist_loss_fn = CompoundLossV2Soft(pos_weight=cfg['pos_weight']).to(device)
specialist_scaler = torch.cuda.amp.GradScaler(enabled=(device.type == 'cuda'))

specialist_ema = copy.deepcopy(specialist_core).to(device).eval()
for p in specialist_ema.parameters():
    p.requires_grad = False

print('specialist params M:', sum(p.numel() for p in specialist_core.parameters() if p.requires_grad) / 1e6)


specialist params M: 3.484457


In [46]:
spec_thresholds = [round(x, 2) for x in np.arange(0.20, 0.86, 0.02)]
spec_log = []
spec_best_iou = -1.0
spec_best_ema_iou = -1.0
spec_start = time.time()

for epoch in range(specialist_cfg['epochs']):
    specialist_model.train()
    loader = loader_spec_warm if epoch < specialist_cfg['warmup_epochs'] else loader_spec_train
    phase = 'WARM' if epoch < specialist_cfg['warmup_epochs'] else 'AUG'

    losses = []
    specialist_optimizer.zero_grad(set_to_none=True)
    pbar = tqdm(loader, desc=f'spec {SPECIALIST_ROVER} ep{epoch:02d} [{phase}]')

    for step, batch in enumerate(pbar):
        images = batch['images'].to(device, non_blocking=True)
        intr = batch['intrinsics'].to(device, non_blocking=True)
        c2c = batch['car2cams'].to(device, non_blocking=True)
        rover_id = batch['rover_id'].to(device, non_blocking=True)
        gt_hard = batch['gt_hard'].to(device, non_blocking=True)
        gt_soft = batch['gt_soft'].to(device, non_blocking=True)
        valid_mask = batch['valid_mask'].to(device, non_blocking=True)

        with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
            logits = specialist_model(images, intr, c2c, rover_id)
            loss, parts = specialist_loss_fn(logits, gt_soft, gt_hard, valid_mask)
            loss = loss / specialist_cfg['grad_accum']

        specialist_scaler.scale(loss).backward()

        if (step + 1) % specialist_cfg['grad_accum'] == 0:
            specialist_scaler.step(specialist_optimizer)
            specialist_scaler.update()
            specialist_optimizer.zero_grad(set_to_none=True)
            update_ema(specialist_ema, specialist_model, specialist_cfg['ema_decay'])

        losses.append(loss.item() * specialist_cfg['grad_accum'])
        if step % 10 == 0:
            pbar.set_postfix(loss=f'{np.mean(losses[-20:]):.3f}', bce=f"{parts['bce']:.2f}")

    specialist_scheduler.step()

    print('start specialist val model @0.5')
    spec_val_iou_05 = evaluate_iou(specialist_model, loader_spec_val, threshold=0.5)

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print('start specialist val ema @0.5')
    spec_val_iou_05_ema = evaluate_iou(specialist_ema, loader_spec_val, threshold=0.5)

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    elapsed = (time.time() - spec_start) / 60

    row = {
        'epoch': epoch,
        'loss': float(np.mean(losses)),
        'val_iou_05': float(spec_val_iou_05),
        'val_iou_05_ema': float(spec_val_iou_05_ema),
        'minutes': float(elapsed),
    }

    print(
        f"spec {SPECIALIST_ROVER} ep{epoch:02d} | loss {np.mean(losses):.3f} | "
        f"val@0.5 {spec_val_iou_05:.4f} | ema@0.5 {spec_val_iou_05_ema:.4f} | "
        f"{elapsed:.1f}m"
    )

    if spec_val_iou_05 > spec_best_iou:
        spec_best_iou = spec_val_iou_05
        torch.save({
            'model_type': 'v4_cleaned_aug_smooth_specialist',
            'specialist_rover': SPECIALIST_ROVER,
            'model': unwrap_model(specialist_model).state_dict(),
            'epoch': epoch,
            'best_iou': float(spec_val_iou_05),
            'best_t': 0.5,
            'rover_vocab': rover_vocab,
            'cfg': cfg,
            'specialist_cfg': specialist_cfg,
        }, SPECIALIST_RUN_DIR / 'best.pt')
        print('  new specialist best model:', spec_val_iou_05)

    if spec_val_iou_05_ema > spec_best_ema_iou:
        spec_best_ema_iou = spec_val_iou_05_ema
        torch.save({
            'model_type': 'v4_cleaned_aug_smooth_specialist',
            'specialist_rover': SPECIALIST_ROVER,
            'model': unwrap_model(specialist_model).state_dict(),
            'ema': specialist_ema.state_dict(),
            'epoch': epoch,
            'best_ema_iou': float(spec_val_iou_05_ema),
            'best_t_ema': 0.5,
            'rover_vocab': rover_vocab,
            'cfg': cfg,
            'specialist_cfg': specialist_cfg,
        }, SPECIALIST_RUN_DIR / 'ema_best.pt')
        print('  new specialist best ema:', spec_val_iou_05_ema)

    spec_log.append(row)
    pd.DataFrame(spec_log).to_csv(SPECIALIST_RUN_DIR / 'log.csv', index=False)

    torch.save({
        'model_type': 'v4_cleaned_aug_smooth_specialist',
        'specialist_rover': SPECIALIST_ROVER,
        'model': unwrap_model(specialist_model).state_dict(),
        'ema': specialist_ema.state_dict(),
        'epoch': epoch,
        'rover_vocab': rover_vocab,
        'cfg': cfg,
        'specialist_cfg': specialist_cfg,
    }, SPECIALIST_RUN_DIR / 'last.pt')


spec darron ep00 [WARM]: 100%|██████████| 24/24 [00:15<00:00,  1.51it/s, bce=1.00, loss=0.970]

start specialist val model @0.5


start specialist val ema @0.5
spec darron ep00 | loss 0.964 | val@0.5 0.4318 | ema@0.5 0.4181 | 0.4m
  new specialist best model: 0.4317520999353866
  new specialist best ema: 0.41808968302145744


spec darron ep01 [AUG]: 100%|██████████| 24/24 [00:16<00:00,  1.44it/s, bce=1.14, loss=0.940]

start specialist val model @0.5


...

start specialist val model @0.5


start specialist val ema @0.5
spec darron ep04 | loss 0.903 | val@0.5 0.3987 | ema@0.5 0.4183 | 2.1m


spec darron ep05 [AUG]: 100%|██████████| 24/24 [00:17<00:00,  1.39it/s, bce=1.09, loss=0.895]

start specialist val model @0.5


start specialist val ema @0.5
spec darron ep05 | loss 0.893 | val@0.5 0.4016 | ema@0.5 0.4192 | 2.5m


In [48]:
import json
import hashlib
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm


In [49]:
# Используем те же индексы камер, что и в ноутбуке.
print('LEFT_IDX =', LEFT_IDX, CAMERA_NAMES[LEFT_IDX])
print('RIGHT_IDX =', RIGHT_IDX, CAMERA_NAMES[RIGHT_IDX])


LEFT_IDX = 2 /side/left/forward
RIGHT_IDX = 3 /side/right/forward


In [50]:
def swap_camera_dim(x, left_idx=LEFT_IDX, right_idx=RIGHT_IDX):
    x = x.clone()
    x[:, [left_idx, right_idx]] = x[:, [right_idx, left_idx]]
    return x


In [51]:
def make_mirror_tta_batch(batch):
    images = batch['images'].clone()
    intr = batch['intrinsics'].clone()
    c2c = batch['car2cams'].clone()

    # 1. Horizontal flip in image plane
    images = torch.flip(images, dims=[-1])

    # 2. Update principal point cx after image flip
    W = images.shape[-1]
    intr[..., 0, 2] = (W - 1) - intr[..., 0, 2]

    # 3. Swap left/right camera slots
    images = swap_camera_dim(images)
    intr = swap_camera_dim(intr)
    c2c = swap_camera_dim(c2c)

    tta_batch = {
        'images': images,
        'intrinsics': intr,
        'car2cams': c2c,
        'rover_id': batch['rover_id'].clone(),
    }

    if 'gt_hard' in batch:
        tta_batch['gt_hard'] = batch['gt_hard']
    if 'gt_soft' in batch:
        tta_batch['gt_soft'] = batch['gt_soft']
    if 'valid_mask' in batch:
        tta_batch['valid_mask'] = batch['valid_mask']
    if 'info_idx' in batch:
        tta_batch['info_idx'] = batch['info_idx']

    return tta_batch


In [52]:
@torch.no_grad()
def predict_probs_with_tta(model, batch):
    images = batch['images'].to(device, non_blocking=True)
    intr = batch['intrinsics'].to(device, non_blocking=True)
    c2c = batch['car2cams'].to(device, non_blocking=True)
    rover_id = batch['rover_id'].to(device, non_blocking=True)

    with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
        logits_plain = model(images, intr, c2c, rover_id).float()
    probs_plain = torch.sigmoid(logits_plain)

    tta_batch = make_mirror_tta_batch(batch)
    images_f = tta_batch['images'].to(device, non_blocking=True)
    intr_f = tta_batch['intrinsics'].to(device, non_blocking=True)
    c2c_f = tta_batch['car2cams'].to(device, non_blocking=True)
    rover_id_f = tta_batch['rover_id'].to(device, non_blocking=True)

    with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
        logits_flip = model(images_f, intr_f, c2c_f, rover_id_f).float()

    # Undo BEV mirror: flip width axis back
    probs_flip = torch.sigmoid(logits_flip)
    probs_flip = torch.flip(probs_flip, dims=[-1])

    probs = 0.5 * (probs_plain + probs_flip)
    return probs


In [53]:
@torch.no_grad()
def evaluate_iou_tta(model, loader, threshold=0.5):
    model.eval()
    inter, union = 0, 0

    for batch in tqdm(loader, desc=f'eval tta @ {threshold:.2f}'):
        probs = predict_probs_with_tta(model, batch).cpu()
        gt_hard = batch['gt_hard']

        valid = (gt_hard != 255)
        gt_b = ((gt_hard == 1) & valid)
        pred = ((probs > threshold) & valid)

        inter += (pred & gt_b).sum().item()
        union += (pred | gt_b).sum().item()

    return inter / max(union, 1)


In [54]:
@torch.no_grad()
def streaming_threshold_sweep_tta(model, loader, thresholds):
    model.eval()
    inter = torch.zeros(len(thresholds), dtype=torch.float64)
    union = torch.zeros(len(thresholds), dtype=torch.float64)

    for batch in tqdm(loader, desc='tta threshold sweep'):
        probs = predict_probs_with_tta(model, batch).cpu()
        gt_hard = batch['gt_hard']

        valid = gt_hard != 255
        gt_b = ((gt_hard == 1) & valid).float()

        for ti, t in enumerate(thresholds):
            pred = ((probs > t) & valid).float()
            inter[ti] += (pred * gt_b).sum().item()
            union[ti] += ((pred + gt_b).clamp(0, 1)).sum().item()

    return {float(t): float(inter[i] / max(union[i].item(), 1.0)) for i, t in enumerate(thresholds)}


In [55]:
CKPT_PATH = RUN_DIR / 'ema_best.pt' if (RUN_DIR / 'ema_best.pt').exists() else RUN_DIR / 'best.pt'
USE_EMA = True

ckpt = torch.load(CKPT_PATH, map_location=device)

tta_model = MultiCamBEVv4CleanAugSmooth(
    num_rover_classes=len(rover_vocab),
    rover_emb_dim=cfg['rover_emb_dim'],
    rover_cond_dim=cfg['rover_cond_dim'],
).to(device)

state = ckpt['ema'] if (USE_EMA and 'ema' in ckpt) else ckpt['model']
missing, unexpected = tta_model.load_state_dict(state, strict=False)

print('checkpoint:', CKPT_PATH)
print('using state:', 'ema' if (USE_EMA and 'ema' in ckpt) else 'model')
print('missing:', len(missing), 'unexpected:', len(unexpected))


checkpoint: runs/v4_cleaned_aug_smooth/ema_best.pt
using state: ema
missing: 0 unexpected: 0


In [56]:
tta_thresholds = [round(x, 2) for x in np.arange(0.20, 0.86, 0.02)]

tta_iou_by_t = streaming_threshold_sweep_tta(tta_model, loader_val, tta_thresholds)
tta_best_t, tta_best_iou = max(tta_iou_by_t.items(), key=lambda kv: kv[1])

print('best TTA threshold:', tta_best_t)
print('best TTA IoU:', tta_best_iou)

for t, iou in tta_iou_by_t.items():
    marker = ' <- best' if t == tta_best_t else ''
    print(f't={t:.2f}  iou={iou:.4f}{marker}')


tta threshold sweep: 100%|██████████| 28/28 [00:20<00:00,  1.37it/s]

best TTA threshold: 0.56
best TTA IoU: 0.5103843586320221
t=0.20  iou=0.4600
t=0.22  iou=0.4642
t=0.24  iou=0.4679
t=0.26  iou=0.4713
t=0.28  iou=0.4743
t=0.30  iou=0.4773
t=0.32  iou=0.4802
t=0.34  iou=0.4827
t=0.36  iou=0.4852
t=0.38  iou=0.4881
t=0.40  iou=0.4909
t=0.42  iou=0.4938
t=0.44  iou=0.4968
t=0.46  iou=0.5003
t=0.48  iou=0.5033
t=0.50  iou=0.5060
t=0.52  iou=0.5083
t=0.54  iou=0.5097
t=0.56  iou=0.5104 <- best
t=0.58  iou=0.5104
t=0.60  iou=0.5092
t=0.62  iou=0.5077
t=0.64  iou=0.5058
t=0.66  iou=0.5035
t=0.68  iou=0.5005
t=0.70  iou=0.4968
t=0.72  iou=0.4902
t=0.74  iou=0.4794
t=0.76  iou=0.4626
t=0.78  iou=0.4419
t=0.80  iou=0.4142
t=0.82  iou=0.3775
t=0.84  iou=0.3331


In [57]:
# Для сравнения можно быстро посмотреть @0.5 без TTA и с TTA
plain_iou_05 = evaluate_iou(tta_model, loader_val, threshold=0.5)
tta_iou_05 = evaluate_iou_tta(tta_model, loader_val, threshold=0.5)

print('plain @0.5:', plain_iou_05)
print('tta   @0.5:', tta_iou_05)


eval tta @ 0.50: 100%|██████████| 28/28 [00:19<00:00,  1.41it/s]

plain @0.5: 0.5311014614078923
tta   @0.5: 0.5059987551253251


In [58]:
# Если ещё нет загруженной eval-модели, используем tta_model из прошлых ячеек.
# Иначе можно подставить свою model_eval.
eval_model = tta_model
eval_model.eval()

rover_thresholds = [round(x, 2) for x in np.arange(0.20, 0.86, 0.02)]


In [59]:
@torch.no_grad()
def collect_val_plain_tta_probs_with_meta(model, loader, info_df):
    plain_probs_list = []
    tta_probs_list = []
    gt_list = []
    rows = []

    for batch in tqdm(loader, desc='collect plain + tta val probs'):
        images = batch['images'].to(device, non_blocking=True)
        intr = batch['intrinsics'].to(device, non_blocking=True)
        c2c = batch['car2cams'].to(device, non_blocking=True)
        rover_id = batch['rover_id'].to(device, non_blocking=True)

        with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
            logits_plain = model(images, intr, c2c, rover_id).float()
        probs_plain = torch.sigmoid(logits_plain).cpu()

        probs_tta = predict_probs_with_tta(model, batch).cpu()

        plain_probs_list.append(probs_plain.to(torch.float16))
        tta_probs_list.append(probs_tta.to(torch.float16))
        gt_list.append(batch['gt_hard'].cpu().to(torch.uint8))

        idxs = batch['info_idx'].cpu().tolist()
        for info_idx in idxs:
            row = info_df.iloc[int(info_idx)]
            rows.append({
                'info_idx': int(info_idx),
                'rover': row.get('rover', '__unknown__'),
                'ride_date': row.get('ride_date', ''),
                'message_ts': row.get('message_ts', ''),
            })

    plain_probs = torch.cat(plain_probs_list, dim=0)
    tta_probs = torch.cat(tta_probs_list, dim=0)
    gt_hard = torch.cat(gt_list, dim=0)
    meta = pd.DataFrame(rows)

    return plain_probs, tta_probs, gt_hard, meta


In [60]:
plain_probs_val, tta_probs_val, gt_hard_val, val_meta = collect_val_plain_tta_probs_with_meta(
    eval_model,
    loader_val,
    val_info,
)

print('plain_probs_val:', tuple(plain_probs_val.shape))
print('tta_probs_val:', tuple(tta_probs_val.shape))
print('gt_hard_val:', tuple(gt_hard_val.shape))
display(val_meta.head())


collect plain + tta val probs: 100%|██████████| 28/28 [00:21<00:00,  1.33it/s]

plain_probs_val: (220, 1, 188, 126)
tta_probs_val: (220, 1, 188, 126)
gt_hard_val: (220, 1, 188, 126)


,info_idx,rover,ride_date,message_ts
0,0,shelly,2021-10-07,1633601020133971000
1,1,shelly,2021-10-07,1633599828934016000
2,2,lerita,2022-07-06,1657087270900063000
3,3,baland,2023-08-29,1693315963400083000
4,4,asterio,2025-05-08,1746685802799978000


In [61]:
def iou_for_probs_subset(probs_subset: torch.Tensor, gt_subset: torch.Tensor, threshold: float):
    valid = (gt_subset != 255)
    gt_b = ((gt_subset == 1) & valid)
    pred = ((probs_subset > threshold) & valid)

    inter = (pred & gt_b).sum().item()
    union = (pred | gt_b).sum().item()
    return inter / max(union, 1)


In [62]:
rows = []

for rover in sorted(val_meta['rover'].unique().tolist()):
    mask_np = (val_meta['rover'].values == rover)
    n_samples = int(mask_np.sum())
    mask = torch.from_numpy(mask_np.astype(np.bool_))

    probs_plain_r = plain_probs_val[mask]
    probs_tta_r = tta_probs_val[mask]
    gt_r = gt_hard_val[mask]

    best_plain_t = None
    best_plain_iou = -1.0
    for t in rover_thresholds:
        iou = iou_for_probs_subset(probs_plain_r, gt_r, t)
        if iou > best_plain_iou:
            best_plain_iou = iou
            best_plain_t = float(t)

    best_tta_t = None
    best_tta_iou = -1.0
    for t in rover_thresholds:
        iou = iou_for_probs_subset(probs_tta_r, gt_r, t)
        if iou > best_tta_iou:
            best_tta_iou = iou
            best_tta_t = float(t)

    rows.append({
        'rover': rover,
        'n_val_samples': n_samples,
        'best_plain_t': best_plain_t,
        'best_plain_iou': best_plain_iou,
        'best_tta_t': best_tta_t,
        'best_tta_iou': best_tta_iou,
        'tta_minus_plain': best_tta_iou - best_plain_iou,
        'better_mode': 'tta' if best_tta_iou > best_plain_iou else 'plain',
    })

rover_cmp_df = pd.DataFrame(rows).sort_values(
    ['tta_minus_plain', 'n_val_samples'],
    ascending=[False, False]
).reset_index(drop=True)

display(rover_cmp_df)


,rover,n_val_samples,best_plain_t,best_plain_iou,best_tta_t,best_tta_iou,tta_minus_plain,better_mode
0,susana,1,0.20,0.603889,0.40,0.652328,0.048439,tta
1,leid,1,0.68,0.513012,0.74,0.558263,0.045251,tta
2,ward,5,0.64,0.634202,0.74,0.669539,0.035336,tta
3,hilma,2,0.58,0.741428,0.58,0.773795,0.032367,tta
4,mika,2,0.26,0.560797,0.52,0.590014,0.029216,tta
5,miro,2,0.20,0.535694,0.26,0.557918,0.022224,tta
6,jurita,1,0.70,0.658602,0.68,0.678976,0.020373,tta
7,greben,13,0.22,0.694931,0.20,0.708966,0.014035,tta
8,franchesca,1,0.26,0.645619,0.36,0.653477,0.007858,tta
9,arlim,2,0.66,0.704239,0.60,0.710250,0.006011,tta


In [63]:
print('Rovers where TTA is better:')
display(rover_cmp_df[rover_cmp_df['tta_minus_plain'] > 0])

print('Rovers where plain is better:')
display(rover_cmp_df[rover_cmp_df['tta_minus_plain'] <= 0])


Rovers where TTA is better:


,rover,n_val_samples,best_plain_t,best_plain_iou,best_tta_t,best_tta_iou,tta_minus_plain,better_mode
0,susana,1,0.20,0.603889,0.40,0.652328,0.048439,tta
1,leid,1,0.68,0.513012,0.74,0.558263,0.045251,tta
2,ward,5,0.64,0.634202,0.74,0.669539,0.035336,tta
3,hilma,2,0.58,0.741428,0.58,0.773795,0.032367,tta
4,mika,2,0.26,0.560797,0.52,0.590014,0.029216,tta
5,miro,2,0.20,0.535694,0.26,0.557918,0.022224,tta
6,jurita,1,0.70,0.658602,0.68,0.678976,0.020373,tta
7,greben,13,0.22,0.694931,0.20,0.708966,0.014035,tta
8,franchesca,1,0.26,0.645619,0.36,0.653477,0.007858,tta
9,arlim,2,0.66,0.704239,0.60,0.710250,0.006011,tta


Rovers where plain is better:


,rover,n_val_samples,best_plain_t,best_plain_iou,best_tta_t,best_tta_iou,tta_minus_plain,better_mode
11,kurt,1,0.20,0.000000,0.20,0.000000,0.000000,plain
12,helen,1,0.74,0.565225,0.72,0.561270,-0.003955,plain
13,lucky,5,0.60,0.815983,0.66,0.811260,-0.004724,plain
14,floi,1,0.36,0.417474,0.24,0.412091,-0.005383,plain
15,onipa,4,0.58,0.680506,0.46,0.674155,-0.006351,plain
16,stelard,1,0.58,0.708895,0.64,0.702511,-0.006383,plain
17,asterio,1,0.30,0.476287,0.24,0.469891,-0.006395,plain
18,litov,1,0.42,0.620238,0.42,0.613464,-0.006774,plain
19,nikena,1,0.60,0.500511,0.64,0.492825,-0.007686,plain
20,kynde,4,0.82,0.521374,0.80,0.513482,-0.007892,plain
